In [ ]:
import torch

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

from torch import nn
from torchinfo import summary

from tqdm.auto import tqdm
from timeit import default_timer as timer

from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

In [ ]:
train_directory = "dataset/divided/train/"
val_directory = "dataset/divided/val/"
test_directory = "dataset/divided/test/"

In [ ]:
transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((360, 360)),
    transforms.RandomRotation(
        degrees=40,
        fill=125
    ),
    transforms.RandomPerspective(
        distortion_scale=0.6,
        p=1.0,
        fill=125
    ),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((360, 360)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = ImageFolder(
    root=train_directory,
    transform=transform_train
)

val_dataset = ImageFolder(
    root=val_directory,
    transform=transform_test
)

test_dataset = ImageFolder(
    root=test_directory,
    transform=transform_test
)

In [ ]:
train_dataset = ImageFolder(train_directory, transform_train)
val_dataset = ImageFolder(val_directory, transform_test)
test_dataset = ImageFolder(test_directory, transform_test)

In [ ]:
imagen, etiqueta = train_dataset[50]

plt.imshow(
    imagen.permute(1, 2, 0),
    cmap="gray"
)

plt.title(
    f"Clase: {train_dataset.classes[etiqueta]}"
)

plt.axis("off")
plt.show()

In [ ]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

len(train_dataloader), len(val_dataloader), len(test_dataloader)

In [ ]:
class TinyVGG(nn.Module):

    def __init__(
        self,
        input_shape: int,
        hidden_units: int,
        output_shape: int
    ):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                in_features=hidden_units * 87 * 87,
                out_features=output_shape
            )
        )

    def forward(self, x):

        x = self.conv_block_1(x)
        # print(x.shape)

        x = self.conv_block_2(x)
        # print(x.shape)

        x = self.classifier(x)
        # print(x.shape)

        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

model_0 = TinyVGG(
    input_shape=1,
    hidden_units=8,
    output_shape=len(train_dataset.classes)
).to(device)

model_0

In [ ]:
image_batch, label_batch = next(
    iter(train_dataloader)
)

print("Forma de las imágenes:")
print(image_batch.shape)

print("Forma de las etiquetas:")
print(label_batch.shape)

In [ ]:
model_0(image_batch.to(device))

In [ ]:
summary(
    model_0,
    input_size=(1, 1, 360, 360)
)